## helper code

In [9]:
def to_um(value_str):
    units = {
        'nm': 1e-3,   # nanometers to micrometers
        'um': 1,      # micrometers to micrometers
        'mm': 1e3,    # millimeters to micrometers
        'cm': 1e4,    # centimeters to micrometers
        'm':  1e6     # meters to micrometers
    }
    
    # Clean and split the input
    parts = value_str.strip().lower().split()
    if len(parts) != 2:
        raise ValueError("Input must be in the form '<number> <unit>'")

    number, unit = parts
    if unit not in units:
        raise ValueError(f"Unsupported unit: {unit}")
    
    return float(number) * units[unit]

# General import

In [10]:
%load_ext autoreload
%autoreload 2
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
%metal_heading First Attempt to design a qubit

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Layout

In [11]:
from qiskit_metal.qlibrary.qubits.transmon_pocket_6 import TransmonPocket6
from qiskit_metal.qlibrary.qubits.transmon_cross import TransmonCross
from qiskit_metal.qlibrary.qubits.transmon_cross_fl import TransmonCrossFL

from qiskit_metal.qlibrary.couplers.tunable_coupler_01 import TunableCoupler01

from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

from qiskit_metal.qlibrary.lumped.cap_n_interdigital import CapNInterdigital
from qiskit_metal.qlibrary.couplers.cap_n_interdigital_tee import CapNInterdigitalTee
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.launchpad_wb_coupled import LaunchpadWirebondCoupled
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround

from qiskit_metal.qlibrary.qubits.JJ_Manhattan import jj_manhattan

In [12]:
design = metal.designs.DesignPlanar()
gui = metal.MetalGUI(design)

In [13]:
design.overwrite_enabled = True
design.chips.main

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '0.0mm',
  'center_z': '0.0mm',
  'size_x': '9mm',
  'size_y': '6mm',
  'size_z': '-750um',
  'sample_holder_top': '890um',
  'sample_holder_bottom': '1650um'}}

In [14]:
design._chips['main']['size']['size_x'] = '5mm'
design._chips['main']['size']['size_y'] = '5mm'
design.variables['cpw_width'] = '10 um'
design.variables['cpw_gap'] = '6 um'

# The Qubit

In [15]:
TransmonCross.get_template_options(design)

{'pos_x': '0.0um',
 'pos_y': '0.0um',
 'orientation': '0.0',
 'chip': 'main',
 'layer': '1',
 'connection_pads': {},
 '_default_connection_pads': {'connector_type': '0',
  'claw_length': '30um',
  'ground_spacing': '5um',
  'claw_width': '10um',
  'claw_gap': '6um',
  'connector_location': '0'},
 'cross_width': '20um',
 'cross_length': '200um',
 'cross_gap': '20um',
 'hfss_inductance': '10nH',
 'hfss_capacitance': 0,
 'hfss_resistance': 0,
 'hfss_mesh_kw_jj': 7e-06,
 'q3d_inductance': '10nH',
 'q3d_capacitance': 0,
 'q3d_resistance': 0,
 'q3d_mesh_kw_jj': 7e-06,
 'gds_cell_name': 'my_other_junction'}

In [16]:
xmon_options = dict(
    pos_x = '1.0mm',
    pos_y = '4.25mm',
    orientation = '270',
    cross_width = '100um',
    connection_pads=dict(
        connector_1 = dict(connector_location = '90', connector_type = '0')
    ),
)

Qubit_1 = TransmonCross(design, 'Qubit_1', options=xmon_options)

gui.rebuild()
gui.autoscale()

# The Readout line

### Readout launchpad

In [17]:
output = LaunchpadWirebondCoupled(design, 'output', options = dict(pos_x='2500um', pos_y='260um', orientation='90', lead_length='30um'))
input = LaunchpadWirebondCoupled(design, 'input', options = dict(pos_x='2500um', pos_y='4740um', orientation='270', lead_length='30um'))
gui.rebuild()
gui.autoscale()


### Readout bus

In [18]:
IObus = RouteStraight(design,'IObus',options=Dict(pin_inputs=Dict(
start_pin=Dict(
        component = 'input',
        pin = 'tie'),
    end_pin=Dict(
        component = 'output',
        pin = 'tie')
)))
gui.rebuild()
gui.autoscale()

# The resonator

### helper functions

In [20]:
pos_ro_x = 2500
cpw_width = 10
epsilon = 6.3
fillet='74.99um'
cpw_options = Dict(
    lead=Dict(
        start_straight='200um',
        end_straight='200um'
    ),
    fillet=fillet,
    meander=Dict(spacing="150um")
)


def pos_from_offset(offset):
    return pos_ro_x + offset

def quarter_wave_length(frequency_ghz, epsilon_eff):
    """input in GHz, output in um"""
    c = 3e8
    frequency_hz = frequency_ghz * 1e9
    wavelength = c / (frequency_hz * (epsilon_eff ** 0.5))
    quarter_wavelength = wavelength / 4
    return quarter_wavelength * 1e6

def generate_readout_frequencies(center_freq_ghz=7.4, spacing_mhz=30, num_qubits=6):
    start_freq = center_freq_ghz - (spacing_mhz * (num_qubits - 1) / 2) / 1000
    return [round(start_freq + i * spacing_mhz / 1000, 6) for i in range(num_qubits)]

def connect(cpw_name: str, pin1_comp_name: str, pin1_comp_pin: str, pin2_comp_name: str, pin2_comp_pin: str,
            length: str, asymmetry='0 um'):
    """Connect two pins with a CPW."""
    myoptions = Dict(
        pin_inputs=Dict(
            start_pin=Dict(
                component=pin1_comp_name,
                pin=pin1_comp_pin),
            end_pin=Dict(
                component=pin2_comp_name,
                pin=pin2_comp_pin)),
        total_length=length)
    myoptions.update(cpw_options)
    myoptions.meander.asymmetry = asymmetry
    return RouteMeander(design, cpw_name, myoptions)

frequencies = generate_readout_frequencies()

length_um_list = [quarter_wave_length(freq, epsilon) for freq in frequencies]

### Resonator

In [21]:
offset_ro_x_1 = -40
pos_x_cp1 = pos_from_offset(offset_ro_x_1)
pos_y_cp1 = 4250

coupling_pin_1 = OpenToGround(design, 'coupling_pin_1', options=dict(
    pos_x = f"{pos_x_cp1}um",
    pos_y = f"{pos_y_cp1}um",
    orientation = "270.0"
))

asym = 0
cpw1 = connect('cpw1', 'Qubit_1', 'connector_1', 'coupling_pin_1', 'open', '4037um', f'+{asym}um')

gui.rebuild()
gui.autoscale()